# Fase A — EDA & Profiling Trip History (Jan–Mar 2026)

**Tujuan** — memetakan kebutuhan data quality pada batch ingest:
1. Profil 7 file mentah: jumlah baris, rentang tanggal, nulls, duplikat `ride_id`.
2. Cek **konsistensi skema** antar bulan (kolom & tipe) → dasar handling schema evolution di staging.
3. Deteksi nilai anomali yang menjadi **DQ test**: durasi tidak wajar, koordinat di luar NYC, dll.
4. Rekap hasil ke `data/data_size_citibike.md` (bahan slide "alasan pemilihan data").

**Sumber:** `data/raw/20260{1,2,3}-citibike-tripdata_*.csv`

In [ ]:
# ============================================================
# 1. Konfigurasi & setup
# ============================================================
import os
import glob
from pathlib import Path

import polars as pl
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Path dataset (folder di-mount konsisten dengan struktur repo)
DATA_RAW = Path("../data/raw").resolve()
FILES = sorted(glob.glob(str(DATA_RAW / "2026*-citibike-tripdata_*.csv")))

# Ambang validasi koordinat NYC (asumsi desain)
NYC_BBOX = {"lat_min": 40.47, "lat_max": 40.92, "lng_min": -74.27, "lng_max": -73.60}

print(f"Jumlah file  : {len(FILES)}")
print(f"Folder       : {DATA_RAW}")
for f in FILES:
    print(f"  - {Path(f).name}")

Jumlah file  : 7
Folder       : C:\Users\sator\OneDrive\Desktop\Purwadhika\final-project-citibike\data\raw
  - 202601-citibike-tripdata_1.csv
  - 202601-citibike-tripdata_2.csv
  - 202602-citibike-tripdata_1.csv
  - 202602-citibike-tripdata_2.csv
  - 202603-citibike-tripdata_1.csv
  - 202603-citibike-tripdata_2.csv
  - 202603-citibike-tripdata_3.csv


In [ ]:
# ============================================================
# 2. Profil per file: baris, kolom, ukuran, & konsistensi skema
# ============================================================
# Baca header tiap file untuk membandingkan skema lintas bulan
# (menangkap schema evolution antar periode data).
schemas = {}
for f in FILES:
    with open(f, "r", encoding="utf-8") as fh:
        header = fh.readline().strip().split(",")
    # baris data (sudah diketahui dari wc -l sebelumnya, tapi hitung cepat ulang)
    schemas[Path(f).name] = header

schema_df = pd.DataFrame(
    [{"file": k, "n_cols": len(v), "cols": "|".join(v)} for k, v in schemas.items()]
)
display(schema_df)

# Normalisasi: cek apakah semua file punya kolom yang sama
col_sets = {frozenset(v) for v in schemas.values()}
print(f"\nJumlah set kolom unik antar file: {len(col_sets)}")
if len(col_sets) == 1:
    print("Semua file konsisten — tidak ada schema evolution lintas bulan ini.")
else:
    print("TERDAPAT perbedaan skema! Detail:")
    base = set(schemas[FILES[0].split("\\")[-1]])
    for name, cols in schemas.items():
        diff = set(cols) ^ base
        if diff:
            print(f"  {name}: beda = {diff}")

,file,n_cols,cols
0,202601-citibike-tripdata_1.csv,13,ride_id|rideable_type|started_at|ended_at|star...
1,202601-citibike-tripdata_2.csv,13,ride_id|rideable_type|started_at|ended_at|star...
2,202602-citibike-tripdata_1.csv,13,ride_id|rideable_type|started_at|ended_at|star...
3,202602-citibike-tripdata_2.csv,13,ride_id|rideable_type|started_at|ended_at|star...
4,202603-citibike-tripdata_1.csv,13,ride_id|rideable_type|started_at|ended_at|star...
5,202603-citibike-tripdata_2.csv,13,ride_id|rideable_type|started_at|ended_at|star...
6,202603-citibike-tripdata_3.csv,13,ride_id|rideable_type|started_at|ended_at|star...



Jumlah set kolom unik antar file: 1
Semua file konsisten — tidak ada schema evolution lintas bulan ini.


In [4]:
# ============================================================
# 3. Baca seluruh data (Polars — hemat memori utk ~7 jt baris)
# ============================================================
# Semua kolom dibaca sebagai string dulu, lalu di-cast eksplisit.
# Tujuannya: kendali tipe penuh & deteksi nilai yang gagal cast
# (calon rejected rows di stg_trips).
COLUMNS = [
    "ride_id", "rideable_type", "started_at", "ended_at",
    "start_station_name", "start_station_id",
    "end_station_name", "end_station_id",
    "start_lat", "start_lng", "end_lat", "end_lng",
    "member_casual",
]

# schema = dict {kolom: tipe} — paksa semua string agar konsisten lintas file
schema_all_utf8 = {c: pl.Utf8 for c in COLUMNS}
lf = pl.scan_csv(FILES, schema=schema_all_utf8)
df = lf.collect()
print(f"Baris total : {df.height:,}")
print(f"Kolom       : {df.width}")
print(f"Memori      : {df.estimated_size('mb'):.0f} MB")
df.head(3)

Baris total : 5,981,588
Kolom       : 13
Memori      : 1038 MB


ride_id,rideable_type,started_at,ended_at,start_station_name,start_station_id,end_station_name,end_station_id,start_lat,start_lng,end_lat,end_lng,member_casual
str,str,str,str,str,str,str,str,str,str,str,str,str
"""85744AF35D7F2DF5""","""electric_bike""","""2026-01-02 05:36:24.539""","""2026-01-02 05:42:21.153""","""W 42 St & 8 Ave""","""6602.05""","""E 58 St & Madison Ave""","""6839.04""","""40.7575699""","""-73.99098507""","""40.76302594280519""","""-73.97209525108337""","""member"""
"""9D18958E5788880B""","""electric_bike""","""2026-01-02 15:15:11.915""","""2026-01-02 15:18:40.462""","""Division St & Bowery""","""5311.08""","""Clinton St & Grand St""","""5303.06_""","""40.71419""","""-73.99673""","""40.71573792143338""","""-73.98698994965027""","""member"""
"""B050891B7B009EE5""","""electric_bike""","""2026-01-12 10:12:51.453""","""2026-01-12 10:16:55.731""","""Broadway & 31 St""","""6789.08""","""35 Ave & 37 St""","""6563.12""","""40.76194""","""-73.92513""","""40.7557327""","""-73.9236611""","""member"""


In [5]:
# ============================================================
# 4. Casting tipe + deteksi baris gagal parse
#    (menjadi dasar pemisahan stg_trips vs stg_trips_rejected)
# ============================================================
df_cast = df.with_columns([
    pl.col("started_at").str.to_datetime(strict=False).alias("started_dt"),
    pl.col("ended_at").str.to_datetime(strict=False).alias("ended_dt"),
    pl.col("start_lat").cast(pl.Float64, strict=False).alias("start_lat_f"),
    pl.col("start_lng").cast(pl.Float64, strict=False).alias("start_lng_f"),
    pl.col("end_lat").cast(pl.Float64, strict=False).alias("end_lat_f"),
    pl.col("end_lng").cast(pl.Float64, strict=False).alias("end_lng_f"),
])

# Ringkasan kegagalan cast per kolom
cast_fail = {
    "started_at": df_cast.filter(pl.col("started_dt").is_null()).height,
    "ended_at": df_cast.filter(pl.col("ended_dt").is_null()).height,
    "start_lat": df_cast.filter(pl.col("start_lat_f").is_null()).height,
    "start_lng": df_cast.filter(pl.col("start_lng_f").is_null()).height,
    "end_lat": df_cast.filter(pl.col("end_lat_f").is_null()).height,
    "end_lng": df_cast.filter(pl.col("end_lng_f").is_null()).height,
}
print("Kegagalan parse (null setelah cast, padahal aslinya terisi):")
for k, v in cast_fail.items():
    print(f"  {k:<12}: {v:,} baris  ({v/df.height*100:.4f}%)")

# Cek apakah ada baris asli yang memang kosong
print("\nNull asli per kolom (string kosong di file):")
null_raw = {
    c: df.filter((pl.col(c).is_null()) | (pl.col(c).str.len_chars() == 0)).height
    for c in ["started_at", "ended_at", "start_lat", "start_lng", "end_lat", "end_lng"]
}
for k, v in null_raw.items():
    print(f"  {k:<12}: {v:,} baris")

Kegagalan parse (null setelah cast, padahal aslinya terisi):
  started_at  : 0 baris  (0.0000%)
  ended_at    : 0 baris  (0.0000%)
  start_lat   : 4,093 baris  (0.0684%)
  start_lng   : 4,093 baris  (0.0684%)
  end_lat     : 25,966 baris  (0.4341%)
  end_lng     : 25,966 baris  (0.4341%)

Null asli per kolom (string kosong di file):
  started_at  : 0 baris
  ended_at    : 0 baris
  start_lat   : 4,093 baris
  start_lng   : 4,093 baris
  end_lat     : 25,966 baris
  end_lng     : 25,966 baris


In [6]:
# ============================================================
# 5. Profil inti: rentang waktu, duplikat, durasi, member type
# ============================================================
df_p = df_cast.with_columns([
    (pl.col("ended_dt") - pl.col("started_dt")).dt.total_seconds().alias("durasi_detik"),
    pl.col("started_dt").dt.date().alias("tanggal"),
])

print("=== Rentang waktu trip ===")
print(f"  Mulai  : {df_p['started_dt'].min()}")
print(f"  Selesai: {df_p['ended_dt'].max()}")

print("\n=== Duplikasi ride_id ===")
n_total = df_p.height
n_unique = df_p["ride_id"].n_unique()
print(f"  Total baris        : {n_total:,}")
print(f"  ride_id unik       : {n_unique:,}")
print(f"  Duplikat (baris)   : {n_total - n_unique:,}  ({(n_total-n_unique)/n_total*100:.4f}%)")

print("\n=== Distribusi durasi trip (detik) ===")
dur = df_p["durasi_detik"]
print(dur.describe())

print("\n=== rideable_type & member_casual ===")
print(df_p.group_by("rideable_type").len().sort("len", descending=True))
print(df_p.group_by("member_casual").len().sort("len", descending=True))

=== Rentang waktu trip ===
  Mulai  : 2025-12-30 23:30:09.507000
  Selesai: 2026-03-31 23:59:59.500000

=== Duplikasi ride_id ===
  Total baris        : 5,981,588
  ride_id unik       : 5,981,588
  Duplikat (baris)   : 0  (0.0000%)

=== Distribusi durasi trip (detik) ===
shape: (9, 2)
┌────────────┬────────────┐
│ statistic  ┆ value      │
│ ---        ┆ ---        │
│ str        ┆ f64        │
╞════════════╪════════════╡
│ count      ┆ 5.981588e6 │
│ null_count ┆ 0.0        │
│ mean       ┆ 710.453382 │
│ std        ┆ 2027.89486 │
│ min        ┆ 4.0        │
│ 25%        ┆ 292.0      │
│ 50%        ┆ 482.0      │
│ 75%        ┆ 811.0      │
│ max        ┆ 93596.0    │
└────────────┴────────────┘

=== rideable_type & member_casual ===
shape: (2, 2)
┌───────────────┬─────────┐
│ rideable_type ┆ len     │
│ ---           ┆ ---     │
│ str           ┆ u32     │
╞═══════════════╪═════════╡
│ electric_bike ┆ 4319924 │
│ classic_bike  ┆ 1661664 │
└───────────────┴─────────┘
shape: (2, 2)
┌──

In [7]:
# ============================================================
# 6. DQ checks §8.1 — durasi negatif / ekstrem & koordinat
# ============================================================
print("=== 6a. Durasi tidak wajar ===")
neg = df_p.filter(pl.col("durasi_detik") < 0)
print(f"  Durasi NEGATIF (ended < started) : {neg.height:,} baris")

# Ambang ekstrem: > 24 jam (86.400 dtk) → indikasi salah catat
extreme = df_p.filter(pl.col("durasi_detik") > 86400)
print(f"  Durasi > 24 jam                 : {extreme.height:,} baris")
if extreme.height > 0:
    print(f"    maks: {extreme['durasi_detik'].max():,.0f} dtk (~{extreme['durasi_detik'].max()/3600:.1f} jam)")

print("\n=== 6b. Koordinat di luar bounding box NYC ===")
lat_ok = (pl.col("start_lat_f").is_between(NYC_BBOX["lat_min"], NYC_BBOX["lat_max"])) & \
         (pl.col("end_lat_f").is_between(NYC_BBOX["lat_min"], NYC_BBOX["lat_max"]))
lng_ok = (pl.col("start_lng_f").is_between(NYC_BBOX["lng_min"], NYC_BBOX["lng_max"])) & \
         (pl.col("end_lng_f").is_between(NYC_BBOX["lng_min"], NYC_BBOX["lng_max"]))
out_of_bbox = df_p.filter((~lat_ok) | (~lng_ok))
print(f"  Trip dengan koord di luar bbox (termasuk null): {out_of_bbox.height:,} baris ({(out_of_bbox.height/df_p.height)*100:.3f}%)")
if out_of_bbox.height > 0:
    print(out_of_bbox.select([
        pl.col("start_lat_f"), pl.col("start_lng_f"),
        pl.col("end_lat_f"), pl.col("end_lng_f"),
        pl.col("start_station_name"), pl.col("end_station_name"),
    ]).head(5))

# Bounding box kita konservatif — cek apakah ada stasiun resmi di luar NYC (NJ, dsb.)
print("\n=== 6c. Trip yang start & end-nya null koordinat ===")
both_null = df_p.filter(pl.col("start_lat_f").is_null() & pl.col("end_lat_f").is_null())
print(f"  start & end lat sama-sama null : {both_null.height:,} baris")

=== 6a. Durasi tidak wajar ===
  Durasi NEGATIF (ended < started) : 0 baris
  Durasi > 24 jam                 : 1,956 baris
    maks: 93,596 dtk (~26.0 jam)

=== 6b. Koordinat di luar bounding box NYC ===
  Trip dengan koord di luar bbox (termasuk null): 0 baris (0.000%)

=== 6c. Trip yang start & end-nya null koordinat ===
  start & end lat sama-sama null : 806 baris


In [8]:
# ============================================================
# 6d. (koreksi) Koordinat — pisahkan null vs di luar bbox
#     Catatan: is_between(null) = null → filter di atas melewatkannya.
# ============================================================
start_null = df_p.filter(pl.col("start_lat_f").is_null() | pl.col("start_lng_f").is_null())
end_null = df_p.filter(pl.col("end_lat_f").is_null() | pl.col("end_lng_f").is_null())

in_bbox = (
    pl.col("start_lat_f").is_between(NYC_BBOX["lat_min"], NYC_BBOX["lat_max"])
    & pl.col("start_lng_f").is_between(NYC_BBOX["lng_min"], NYC_BBOX["lng_max"])
    & pl.col("end_lat_f").is_between(NYC_BBOX["lat_min"], NYC_BBOX["lat_max"])
    & pl.col("end_lng_f").is_between(NYC_BBOX["lng_min"], NYC_BBOX["lng_max"])
)
# is_between menghasilkan null bila input null → force ke False
out_bbox = df_p.filter(~in_bbox.fill_null(False))

print(f"Start koordinat null/tidak valid : {start_null.height:,} baris ({(start_null.height/df_p.height)*100:.3f}%)")
print(f"End koordinat null/tidak valid   : {end_null.height:,} baris ({(end_null.height/df_p.height)*100:.3f}%)")
print(f"Trip dgn koord NON-null di luar NYC bbox : {out_bbox.height - (start_null.height + end_null.height):,} baris")

# Sample nilai koordinat non-null yang di luar bbox (kalau ada)
nonnull_out = df_p.filter(
    in_bbox.fill_null(False).not_()
    & pl.col("start_lat_f").is_not_null()
    & pl.col("end_lat_f").is_not_null()
)
if nonnull_out.height > 0:
    print("\nContoh koordinat di luar bbox:")
    print(nonnull_out.select(["start_station_name", "start_lat_f", "start_lng_f",
                              "end_station_name", "end_lat_f", "end_lng_f"]).head(10))
else:
    print("\nSemua koordinat non-null berada dalam bbox NYC.")

Start koordinat null/tidak valid : 4,093 baris (0.068%)
End koordinat null/tidak valid   : 25,966 baris (0.434%)
Trip dgn koord NON-null di luar NYC bbox : -806 baris

Semua koordinat non-null berada dalam bbox NYC.


In [ ]:
# ============================================================
# 7. Konsistensi station_id (dimensi stasiun utk dim_station)
#    Memeriksa apakah penulisan station_id stabil antar periode.
# ============================================================
s_id = df_p.select([
    pl.col("start_station_id").alias("station_id"),
    pl.col("start_station_name").alias("station_name"),
])
e_id = df_p.select([
    pl.col("end_station_id").alias("station_id"),
    pl.col("end_station_name").alias("station_name"),
])
stasiun = pl.concat([s_id, e_id]).filter(pl.col("station_id").is_not_null() & (pl.col("station_id") != ""))

# Cek: format station_id yang aneh (mengandung titik / underscore / bukan numerik)
format_aneh = stasiun.filter(~pl.col("station_id").str.contains(r"^\d+(\.\d+)?$"))
print(f"Total baris stasiun (start+end, non-null): {stasiun.height:,}")
print(f"station_id dengan format NON-standar     : {format_aneh.height:,}")
if format_aneh.height > 0:
    print(format_aneh.group_by("station_id").agg(pl.len().alias("jumlah")).sort("jumlah", descending=True).head(10))

print("\n=== station_id ↔ nama unik ===")
id_name = (stasiun.group_by(["station_id", "station_name"]).len()
           .sort(["station_id", "len"], descending=[False, True]))
n_id = stasiun["station_id"].n_unique()
n_pairs = id_name.height
print(f"station_id unik          : {n_id:,}")
print(f"Pasangan (id, nama) unik : {n_pairs:,}")

# Apakah ada 1 id dengan banyak nama (indikasi nama berubah / data kotor)?
satu_id_banyak_nama = (id_name.group_by("station_id").agg(pl.len().alias("n_nama"))
                       .filter(pl.col("n_nama") > 1).sort("n_nama", descending=True))
print(f"\nstation_id yang punya >1 nama: {satu_id_banyak_nama.height:,}")
if satu_id_banyak_nama.height > 0:
    print(id_name.filter(pl.col("station_id").is_in(satu_id_banyak_nama["station_id"].head(5))).head(15))

print("\n=== Jumlah stasiun unik keseluruhan ===")
print(f"  {stasiun['station_id'].n_unique():,} stasiun unik (start/end digabung)")
print(f"  {stasiun['station_name'].n_unique():,} nama stasiun unik")

Total baris stasiun (start+end, non-null): 11,932,961
station_id dengan format NON-standar     : 8,186
shape: (10, 2)
┌──────────────┬────────┐
│ station_id   ┆ jumlah │
│ ---          ┆ ---    │
│ str          ┆ u32    │
╞══════════════╪════════╡
│ 5303.06_     ┆ 5995   │
│ 5308.04_     ┆ 507    │
│ 6569.09_     ┆ 374    │
│ 6517.08_     ┆ 338    │
│ SYS016       ┆ 265    │
│ SYS033       ┆ 189    │
│ SYS038       ┆ 90     │
│ JC072        ┆ 69     │
│ HB202        ┆ 44     │
│ Shop Morgan  ┆ 42     │
└──────────────┴────────┘

=== station_id ↔ nama unik ===
station_id unik          : 2,356
Pasangan (id, nama) unik : 2,356

station_id yang punya >1 nama: 0

=== Jumlah stasiun unik keseluruhan ===
  2,356 stasiun unik (start/end digabung)
  2,286 nama stasiun unik


In [10]:
# ============================================================
# 7b. Investigasi station_id non-standar
# ============================================================
# Pola underscore suffix (5303.06_) — kemungkinan id stasiun versi lama
us = format_aneh.filter(pl.col("station_id").str.ends_with("_"))
print(f"station_id ber-akhiran '_' : {us.height:,}")
if us.height > 0:
    print(us.group_by("station_id").agg(pl.len().alias("n")).sort("n", descending=True).head(5))
    # apakah id dasarnya (tanpa _) juga muncul sebagai id normal?
    base = us.select(pl.col("station_id").str.strip_suffix("_").unique().alias("base_id"))
    normal = set(stasiun.filter(pl.col("station_id").str.contains(r"^\d+(\.\d+)?$"))["station_id"].unique())
    overlap = base.filter(pl.col("base_id").is_in(list(normal)))
    print(f"\n  Base id (tanpa _) yang JUGA muncul normal: {overlap.height} dari {base.height}")

# Sisa non-standar selain underscore
lain = format_aneh.filter(~pl.col("station_id").str.ends_with("_"))
print(f"\nNon-standar selain '_' : {lain.height:,} baris")
print(lain.group_by("station_id").agg(pl.len().alias("n")).sort("n", descending=True).head(15))

# Cek baris 'Shop Morgan' — nama salah masuk kolom id?
shop = df_p.filter(pl.col("start_station_id").str.contains("Shop", literal=False) |
                   pl.col("end_station_id").str.contains("Shop", literal=False))
print(f"\nBaris dgn station_id berisi 'Shop': {shop.height}")
if shop.height > 0:
    print(shop.select(["start_station_name", "start_station_id",
                       "end_station_name", "end_station_id", "started_dt"]).head(3))

station_id ber-akhiran '_' : 7,214
shape: (4, 2)
┌────────────┬──────┐
│ station_id ┆ n    │
│ ---        ┆ ---  │
│ str        ┆ u32  │
╞════════════╪══════╡
│ 5303.06_   ┆ 5995 │
│ 5308.04_   ┆ 507  │
│ 6569.09_   ┆ 374  │
│ 6517.08_   ┆ 338  │
└────────────┴──────┘

  Base id (tanpa _) yang JUGA muncul normal: 4 dari 4

Non-standar selain '_' : 972 baris
shape: (15, 2)
┌────────────┬─────┐
│ station_id ┆ n   │
│ ---        ┆ --- │
│ str        ┆ u32 │
╞════════════╪═════╡
│ SYS016     ┆ 265 │
│ SYS033     ┆ 189 │
│ SYS038     ┆ 90  │
│ JC072      ┆ 69  │
│ HB202      ┆ 44  │
│ …          ┆ …   │
│ JC099      ┆ 12  │
│ JC014      ┆ 11  │
│ JC032      ┆ 10  │
│ HB603      ┆ 10  │
│ HB502      ┆ 10  │
└────────────┴─────┘

Baris dgn station_id berisi 'Shop': 41
shape: (3, 5)
┌───────────────────────────┬──────────────────┬──────────────────┬────────────────┬───────────────┐
│ start_station_name        ┆ start_station_id ┆ end_station_name ┆ end_station_id ┆ started_dt    │
│ ---       

In [11]:
# ============================================================
# 7c. Apakah '5303.06_' dan '5303.06' = stasiun yang sama?
#     (menentukan perlu/tidaknya normalisasi id di staging)
# ============================================================
def nama_untuk(id_val: str):
    return (stasiun.filter(pl.col("station_id") == id_val)
            .select("station_name").unique().to_series().to_list())

for sid in ["5303.06", "5303.06_", "5308.04", "5308.04_", "JC072", "SYS016"]:
    print(f"{sid:<10} -> {nama_untuk(sid)}")

# Baris 'Shop Morgan': cek apakah 'Shop Morgan' juga ada sbg station_name valid
print("\n'Shop Morgan' sbg station_name:", nama_untuk("Shop Morgan"))
print("Jumlah nama stasiun mengandung 'Morgan':",
      stasiun.filter(pl.col("station_name").str.contains("Morgan")).select("station_name").n_unique())

5303.06    -> ['Clinton St & Grand St']
5303.06_   -> ['Clinton St & Grand St']
5308.04    -> ['Metropolitan Ave & Bedford Ave']
5308.04_   -> ['Metropolitan Ave & Bedford Ave ']
JC072      -> ['Morris Canal']
SYS016     -> ['Morgan Bike Mechanics']

'Shop Morgan' sbg station_name: []
Jumlah nama stasiun mengandung 'Morgan': 8


In [12]:
# ============================================================
# 8. Profil temporal: baris per bulan & per hari
#    (utk konfirmasi jangkauan batch & partisi harian datalake)
# ============================================================
df_p = df_p.with_columns([
    pl.col("started_dt").dt.strftime("%Y-%m").alias("bulan"),
    pl.col("started_dt").dt.strftime("%Y-%m-%d").alias("hari"),
])

bulanan = (df_p.group_by("bulan").len().sort("bulan"))
print("=== Trip per bulan (berdasar started_at) ===")
print(bulanan)

# Cek ada trip Des 2025 yang nyangkut (boundary file bulanan)
des = df_p.filter(pl.col("bulan") == "2025-12")
print(f"\nTrip dgn started_at Des 2025 (nyangkut dari bulan sblmnya): {des.height:,} baris")

harian = df_p.group_by("hari").len().sort("hari")
print(f"\n=== Trip per hari ===")
print(f"Rentang tanggal : {harian['hari'].min()} s/d {harian['hari'].max()}")
print(f"Jumlah hari unik: {harian.height}")
print(harian.describe())

# Rata-rata trip/hari per bulan (justifikasi pemilihan bulan di slide)
rata_per_bulan = (
    harian.with_columns(pl.col("hari").str.slice(0, 7).alias("bulan"))
    .group_by("bulan")
    .agg([
        pl.col("len").mean().round(0).alias("rata_trip_per_hari"),
        pl.col("hari").count().alias("jumlah_hari"),
        pl.col("len").sum().alias("total_trip"),
    ])
    .sort("bulan")
)
print("\n=== Rata-rata trip per hari per bulan ===")
print(rata_per_bulan)

=== Trip per bulan (berdasar started_at) ===
shape: (4, 2)
┌─────────┬─────────┐
│ bulan   ┆ len     │
│ ---     ┆ ---     │
│ str     ┆ u32     │
╞═════════╪═════════╡
│ 2025-12 ┆ 266     │
│ 2026-01 ┆ 1816296 │
│ 2026-02 ┆ 1219706 │
│ 2026-03 ┆ 2945320 │
└─────────┴─────────┘

Trip dgn started_at Des 2025 (nyangkut dari bulan sblmnya): 266 baris

=== Trip per hari ===
Rentang tanggal : 2025-12-30 s/d 2026-03-31
Jumlah hari unik: 91
shape: (9, 3)
┌────────────┬────────────┬──────────────┐
│ statistic  ┆ hari       ┆ len          │
│ ---        ┆ ---        ┆ ---          │
│ str        ┆ str        ┆ f64          │
╞════════════╪════════════╪══════════════╡
│ count      ┆ 91         ┆ 91.0         │
│ null_count ┆ 0          ┆ 0.0          │
│ mean       ┆ null       ┆ 65731.736264 │
│ std        ┆ null       ┆ 36063.690871 │
│ min        ┆ 2025-12-30 ┆ 1.0          │
│ 25%        ┆ null       ┆ 42038.0      │
│ 50%        ┆ null       ┆ 60100.0      │
│ 75%        ┆ null       ┆ 9239

In [13]:
# ============================================================
# 8b. Boundary file bulanan: file dipotong by started_at atau ended_at?
#     (menentukan kolom partisi harian di datalake — keputusan desain)
# ============================================================
print("=== Boundary Januari (202601_1) ===")
f_jan = pl.scan_csv(str(DATA_RAW / "202601-citibike-tripdata_1.csv"),
                    schema={c: pl.Utf8 for c in COLUMNS}).collect()
jan_dt = f_jan.with_columns([
    pl.col("started_at").str.to_datetime().alias("s"),
    pl.col("ended_at").str.to_datetime().alias("e"),
])
print(f"started_at min : {jan_dt['s'].min()}")
print(f"started_at max : {jan_dt['s'].max()}")
print(f"ended_at   max : {jan_dt['e'].max()}")

# Berapa trip di file Jan yang ended-nya sudah masuk Feb? (kalau 0 → file by ended_at)
print("\n=== Cek file Maret: trip mulai Feb di dalamnya ===")
f_mar = pl.scan_csv([str(DATA_RAW / "202603-citibike-tripdata_1.csv")],
                    schema={c: pl.Utf8 for c in COLUMNS}).collect()
mar_dt = f_mar.with_columns([
    pl.col("started_at").str.to_datetime().alias("s"),
    pl.col("ended_at").str.to_datetime().alias("e"),
])
print(f"started_at min : {mar_dt['s'].min()}  (kalau < 2026-03-01 → file dipotong by ENDED date)")
print(f"ended_at   min : {mar_dt['e'].min()}")

=== Boundary Januari (202601_1) ===
started_at min : 2025-12-30 23:30:09.507000
started_at max : 2026-01-14 18:59:59.453000
ended_at   max : 2026-01-15 19:52:37.747000

=== Cek file Maret: trip mulai Feb di dalamnya ===
started_at min : 2026-02-28 06:05:20.166000  (kalau < 2026-03-01 → file dipotong by ENDED date)
ended_at   min : 2026-03-01 00:00:00.683000


In [15]:
# (d) Distribusi durasi (log scale)
dur_seri = df_p.filter(pl.col("durasi_detik") <= 7200)["durasi_detik"]  # potong >2 jam utk visual
axes[1, 1].hist(dur_seri.to_numpy(), bins=60, color="#264653", alpha=0.85)
axes[1, 1].set_title("Distribusi Durasi Trip (≤2 jam)")
axes[1, 1].set_xlabel("durasi (detik)")
axes[1, 1].set_ylabel("frekuensi")

Text(645.4594578598483, 0.5, 'frekuensi')

In [16]:
# ============================================================
# 10. Rekap hasil → disimpan ke data/data_size_citibike.md
#     (bahan slide: justifikasi pemilihan Jan–Mar & kondisi data)
# ============================================================
import json

ringkasan = {
    "rentang_data": {
        "started_at_min": str(df_p["started_dt"].min()),
        "started_at_max": str(df_p["started_dt"].max()),
        "bulan_tercakup": bulanan["bulan"].to_list(),
    },
    "total_trip": int(df_p.height),
    "trip_per_bulan": {r["bulan"]: int(r["len"]) for r in bulanan.to_dicts()},
    "trip_per_hari": int(harian["len"].mean()),
    "hari_unik": int(harian.height),
    "duplikat_ride_id": int(df_p.height - df_p["ride_id"].n_unique()),
    "stasiun_unik": int(stasiun["station_id"].n_unique()),
    "dq_temuan": {
        "durasi_negatif": int(df_p.filter(pl.col("durasi_detik") < 0).height),
        "durasi_lebih_24jam": int(df_p.filter(pl.col("durasi_detik") > 86400).height),
        "start_koord_null": int(start_null.height),
        "end_koord_null": int(end_null.height),
        "koord_nonnull_diluar_nyc": 0,
        "station_id_format_nonstandar": int(format_aneh.height),
        "nama_stasiun_spasi_ujung": int(stasiun.filter(pl.col("station_name") != pl.col("station_name").str.strip_chars()).height),
        "trip_des2025_nyangkut": int(des.height),
    },
    "catatan_penting": [
        "File bulanan Citi Bike dipotong berdasarkan ENDED date (file Mar berisi trip mulai 28 Feb).",
        "Partisi harian datalake sebaiknya memakai kolom yang konsisten (started date) + dedup ride_id.",
        "station_id non-standar: suffix '_' (5303.06_ = 5303.06), prefix SYS/JC/HB, dan nama bocor ke kolom id ('Shop Morgan').",
        "Tidak ada duplikat ride_id di dataset ini.",
        "Format durasi sudah non-negatif di sumber (min 4 detik).",
    ],
}

with open("../data/data_size_citibike.md", "w", encoding="utf-8") as fh:
    fh.write("# Ukuran & Profil Dataset Citi Bike (Jan–Mar 2026)\n\n")
    fh.write("> Di-generate otomatis oleh `notebooks/01_eda_trip_history.ipynb` — Fase A EDA.\n\n")
    fh.write("## Ringkasan\n\n")
    fh.write(f"| Metrik | Nilai |\n|---|---|\n")
    fh.write(f"| Total trip | {ringkasan['total_trip']:,} |\n")
    fh.write(f"| Rentang `started_at` | {ringkasan['rentang_data']['started_at_min']} s/d {ringkasan['rentang_data']['started_at_max']} |\n")
    for b in ["2026-01", "2026-02", "2026-03"]:
        fh.write(f"| Trip {b} | {ringkasan['trip_per_bulan'][b]:,} |\n")
    fh.write(f"| Rata-rata trip/hari | {ringkasan['trip_per_hari']:,.0f} |\n")
    fh.write(f"| Hari unik | {ringkasan['hari_unik']} |\n")
    fh.write(f"| Duplikat `ride_id` | {ringkasan['duplikat_ride_id']:,} |\n")
    fh.write(f"| Stasiun unik | {ringkasan['stasiun_unik']:,} |\n")
    fh.write("\n## Temuan DQ (menjadi dasar dbt test di staging)\n\n")
    fh.write("| Check | Jumlah |\n|---|---|\n")
    dq = ringkasan["dq_temuan"]
    fh.write(f"| Durasi negatif | {dq['durasi_negatif']:,} |\n")
    fh.write(f"| Durasi > 24 jam | {dq['durasi_lebih_24jam']:,} |\n")
    fh.write(f"| Start koordinat null | {dq['start_koord_null']:,} |\n")
    fh.write(f"| End koordinat null | {dq['end_koord_null']:,} |\n")
    fh.write(f"| Koordinat non-null di luar NYC | {dq['koord_nonnull_diluar_nyc']:,} |\n")
    fh.write(f"| station_id format non-standar | {dq['station_id_format_nonstandar']:,} |\n")
    fh.write(f"| Nama stasiun dgn spasi di ujung | {dq['nama_stasiun_spasi_ujung']:,} |\n")
    fh.write(f"| Trip Des-2025 nyangkut di file Jan | {dq['trip_des2025_nyangkut']:,} |\n")
    fh.write("\n## Catatan Desain\n\n")
    for c in ringkasan["catatan_penting"]:
        fh.write(f"- {c}\n")

print("Tersimpan ke data/data_size_citibike.md")
ringkasan

Tersimpan ke data/data_size_citibike.md


{'rentang_data': {'started_at_min': '2025-12-30 23:30:09.507000',
  'started_at_max': '2026-03-31 23:58:13.842000',
  'bulan_tercakup': ['2025-12', '2026-01', '2026-02', '2026-03']},
 'total_trip': 5981588,
 'trip_per_bulan': {'2025-12': 266,
  '2026-01': 1816296,
  '2026-02': 1219706,
  '2026-03': 2945320},
 'trip_per_hari': 65731,
 'hari_unik': 91,
 'duplikat_ride_id': 0,
 'stasiun_unik': 2356,
 'dq_temuan': {'durasi_negatif': 0,
  'durasi_lebih_24jam': 1956,
  'start_koord_null': 4093,
  'end_koord_null': 25966,
  'koord_nonnull_diluar_nyc': 0,
  'station_id_format_nonstandar': 8186,
  'nama_stasiun_spasi_ujung': 507,
  'trip_des2025_nyangkut': 266},
 'catatan_penting': ['File bulanan Citi Bike dipotong berdasarkan ENDED date (file Mar berisi trip mulai 28 Feb).',
  'Partisi harian datalake sebaiknya memakai kolom yang konsisten (started date) + dedup ride_id.',
  "station_id non-standar: suffix '_' (5303.06_ = 5303.06), prefix SYS/JC/HB, dan nama bocor ke kolom id ('Shop Morgan')."

## Kesimpulan Fase A → Keputusan Desain

### Yang bersih (tidak perlu DQ ketat)
- **Tidak ada duplikat `ride_id`** → test `unique` tetap dipasang sebagai jaring pengaman, tapi tak akan memicu karantina massal.
- **Tidak ada durasi negatif** (sudah difilter sumber) → test tetap ada, tapi ekspektasi 0.
- **Semua koordinat non-null berada di dalam bounding box NYC** → lat/lng out-of-range bukan isu utama; **null** justru isu utamanya.

### Yang perlu penanganan di staging (`stg_trips` → valid / `stg_trips_rejected`)
| Temuan | Jumlah | Keputusan |
|---|---|---|
| `end_lat/lng` null | 25.966 (0,43%) | Baris tetap **valid** bila `end_station_id` ada (GPS gagal capture); koordinat diisi dari `dim_station`. Null di kedua-duanya + tanpa station → **rejected** |
| `start_lat/lng` null | 4.093 | Sama: valid bila `start_station_id` ada |
| Durasi > 24 jam | 1.956 | Asumsi desain: trip >24 jam dimasukkan rejected (`rejection_reason = 'duration_extreme'`) karena indikasi salah catat |
| station_id non-standar | 8.186 | Normalisasi di staging: strip suffix `_` (`5303.06_` → `5303.06`); id `SYS/JC/HB` dianggap valid (sistem/Jersey City/Hoboken); nama bocor ke kolom id (`Shop Morgan`) → **rejected** |
| Nama stasiun spasi ujung | 507 | `TRIM` nama di staging sebelum masuk `dim_station` |
| Trip Des-2025 di file Jan | 266 | File dipotong by `ended_at` → partisi datalake memakai **started date** + dedup `ride_id` (idempoten) |

### Implikasi arsitektur (untuk Fase C)
1. **Partisi datalake harian** → pakai `DATE(started_at)`; file bulanan di-split per hari by started date.
2. **`dim_station`** dibangun dari pasangan `(station_id, station_name, lat, lng)` yang sudah dinormalisasi + di-dedup (2.356 id).
3. **DQ test batch** (`stg_trips`): `not_null(ride_id, started_at, ended_at)`, `unique(ride_id)`, durasi ≥ 0, dll.
4. Angka-angka ini tersimpan di `data/data_size_citibike.md` untuk dokumentasi & slide.